In [4]:
#!pip install tqdm

In [13]:
import json
import os
import time
from pathlib import Path

import torch
from PIL import Image
from tqdm import tqdm
from transformers import Blip2Processor, Blip2ForConditionalGeneration

In [14]:
SEED = 42

MODEL_ID = "Salesforce/blip2-flan-t5-xl"


META_PATH = Path("subsets/coco_caption_5k/metadata.jsonl")
OUT_DIR = Path("runs/blip2/captioning/coco_caption_5k")
PRED_PATH = OUT_DIR / "preds.jsonl"
CFG_PATH = OUT_DIR / "run_config.json"

In [15]:
def read_jsonl(path: Path):
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                yield json.loads(line)

In [16]:
def count_nonempty_lines(path: Path) -> int:
    if not path.exists():
        return 0
    n = 0
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                n += 1
    return n


In [17]:
def append_jsonl(path: Path, row: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")


def write_json(path: Path, obj: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

In [18]:
def main():
    if not META_PATH.exists():
        raise FileNotFoundError(f"Missing: {META_PATH}")

    torch.manual_seed(SEED)

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA not available. This script is intended to run on the GPU.")

    device = torch.device("cuda")
    dtype = torch.float16  # good default for A100

    print(f"Loading BLIP-2 on GPU: {MODEL_ID}")
    processor = Blip2Processor.from_pretrained(MODEL_ID)

    # Prefer device_map="auto" for big BLIP-2 models on GPU
    model = Blip2ForConditionalGeneration.from_pretrained(
        MODEL_ID,
        torch_dtype=dtype,
        device_map="auto",
        low_cpu_mem_usage=True,
    )
    model.eval()

    gen = {
        "max_new_tokens": 30,
        "num_beams": 5,
        "do_sample": False,
    }

    OUT_DIR.mkdir(parents=True, exist_ok=True)
    write_json(CFG_PATH, {
        "run_name": "blip2_caption_coco5k",
        "model_id": MODEL_ID,
        "task": "captioning",
        "dataset": "coco_caption_5k",
        "seed": SEED,
        "metadata_path": str(META_PATH),
        "preds_path": str(PRED_PATH),
        "device": "cuda",
        "dtype": str(dtype),
        "gen": gen,
        "time_unix": time.time(),
    })

    rows = list(read_jsonl(META_PATH))
    total = len(rows)
    if total != 5000:
        print(f"Warning: expected 5000 metadata rows, found {total}")

    done = count_nonempty_lines(PRED_PATH)
    if done > 0:
        print(f"Resuming: {done} already written -> {PRED_PATH}")

    # Optional: speed on Ampere+ (A100)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

    for i in tqdm(range(done, total), desc="BLIP-2 captioning"):
        ex = rows[i]
        image_id = int(ex["image_id"])
        image_file = ex["image_file"]
        refs = ex.get("captions", [])

        img = Image.open(image_file).convert("RGB")

        # IMPORTANT: processor outputs should go to the same device as the model
        # When using device_map="auto", ensure tensors are on CUDA (works for BLIP-2)
        inputs = processor(images=img, return_tensors="pt")
        inputs = {k: v.to(device, non_blocking=True) for k, v in inputs.items()}

        with torch.inference_mode():
            out_ids = model.generate(**inputs, **gen)

        pred = processor.tokenizer.decode(out_ids[0], skip_special_tokens=True).strip()

        out_row = {
            "example_id": f"coco:{image_id}",
            "dataset": "coco_caption_5k",
            "task": "captioning",
            "image_id": image_id,
            "image_file": image_file,
            "references": refs,
            "prediction": pred,
            "model_id": MODEL_ID,
            "seed": SEED,
            "gen": gen,
            "intervention": {"name": "none"},
        }
        append_jsonl(PRED_PATH, out_row)

    print("DONE")
    print("Predictions:", PRED_PATH)
    print("Config:", CFG_PATH)


if __name__ == "__main__":
    main()


Loading BLIP-2 on GPU: Salesforce/blip2-flan-t5-xl


Loading checkpoint shards: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:03<00:00,  1.60s/it]


Resuming: 337 already written -> runs/blip2/captioning/coco_caption_5k/preds.jsonl


BLIP-2 captioning: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4663/4663 [50:58<00:00,  1.52it/s]

DONE
Predictions: runs/blip2/captioning/coco_caption_5k/preds.jsonl
Config: runs/blip2/captioning/coco_caption_5k/run_config.json
